## Import

In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import os
import duckdb
import pandas as pd

from sindex.metrics.citations import (
    merge_citations_dicts, 
    merge_citations_from_files, 
    merge_citations_from_files_fast,
    combine_citations,
)
from sindex.metrics.mentions import combine_mentions
from sindex.metrics.fairscores import merge_doi_fair_scores_ndjson_files, extrapolate_emdb_fair_scores
from sindex.utils.files import combine_ndjson_files, merge_ndjson_files_in_folder
from sindex.metrics.topics import enhance_topics, restructure_topics_ndjson
from sindex.metrics.batch_jobs import (
    batch_process_metadata_from_slim, 
    create_metadata_table,
    create_citations_table,
    create_mentions_table,
    create_fair_scores_table,
    create_topics_table,
    create_dataset_metrics_table,
    calculate_normalization_factors_subfields_rawDindex,
    calculate_normalization_factors_topics,
    calculate_normalization_factors_subfields,
    create_dataset_index_table,
    create_creators_table,
    create_s_index_identifier_table,
    create_s_index_name_affiliation_table,
    create_s_index_identifier_name_affiliation_table
)

## Citations

### Deduplicate citations from different sources

#### DataCite (DOIs)

In [24]:
mdc_citations = r"D:\pipeline-data\citations\mdc\mdc_citations.ndjson"
oa_citations = r"D:\pipeline-data\citations\openalex\oa_citations.ndjson"
dc_citations = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
citation_files = [mdc_citations, oa_citations, dc_citations]
output_file_doi = r"D:\pipeline-data\citations\doi_citations.ndjson"

In [25]:
merge_citations_from_files_fast(citation_files, output_file_doi)

Starting merge of 3 valid files...
Finished processing 8,861,682 records. Unique: 7,654,129
Writing to D:\pipeline-data\citations\doi_citations.ndjson...
Done!


#### EMDB

In [17]:
mdc_citations = r"I:\pipeline-data\citations\mdc\mdc_citations_emdb.ndjson"
citation_files = [mdc_citations]
output_file_emdb = r"I:\pipeline-data\citations\emdb_citations.ndjson"

In [18]:
merge_citations_from_files_fast(citation_files, output_file_emdb)

Starting merge of 1 valid files...
Finished processing 15,134 records. Unique: 15,134
Writing to I:\pipeline-data\citations\emdb_citations.ndjson...
Done!


### Combine and add placeholder dates when citation date missing

In [3]:
doi = r"D:\pipeline-data\citations\doi_citations.ndjson"
emdb = r"D:\pipeline-data\citations\emdb_citations.ndjson"
file_list = [doi, emdb]
output_path = r"D:\pipeline-data\citations\citations.ndjson"

In [4]:
combine_ndjson_files(file_list, output_path)

Lines processed: 7,600,000
Finished! Total entries saved: 7,669,263


## Mentions

### Combine and add placeholder dates when mention date missing

In [12]:
mock = r"D:\pipeline-data\mentions\mentions_github_mock.ndjson"
file_list = [mock]
output_path = r"D:\pipeline-data\mentions\mentions.ndjson"

In [13]:
combine_mentions(file_list, output_path)

Lines processed: 4,150,000
Finished! Total entries saved: 4,156,510


## FAIR scores

### Merge DOI FAIR score into one file (rename doi field as dataset_id)

In [15]:
fair_scores_directory = r"D:\pipeline-data\fair_scores\fair_scores_doi_files"
doi_fair_scores_path = r"D:\pipeline-data\fair_scores\doi_fair_scores.ndjson"

In [16]:
merge_doi_fair_scores_ndjson_files(fair_scores_directory, doi_fair_scores_path)

Found 4901 files. Starting merge with orjson...
Done! Total lines in 'D:\pipeline-data\fair_scores\doi_fair_scores.ndjson': 49,009,521    


In [17]:
#Add missing 1 DOI (based on F-UJI website score)
new_entry = {
    "dataset_id": "10.57451/lhd.ficxs-2.156223.1",
    "score": 65.00,
    "evaluationDate": "2026-02-02T00:00:00+00:00",
    "metricVersion": "0.8",
    "softwareVersion": "website"
}

file_path = doi_fair_scores_path

with open(file_path, 'a') as f:
    f.write(json.dumps(new_entry) + '\n')

### Extrapolate EMDB fair scores (all the same on the first 10k calculated with F-UJI)

In [20]:
emdb_file_path = r"D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson"
partial_score_file_path = r"D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson"
emdb_fair_scores_path = r"D:\pipeline-data\fair_scores\emdb_fair_scores.ndjson"

In [30]:
extrapolate_emdb_fair_scores(emdb_file_path, partial_score_file_path, emdb_fair_scores_path)

Loading scores from D:\pipeline-data\fair_scores\fair_scores_emdb_partial\fair_scores_emdb_partial.ndjson...
Processing D:\pipeline-data\records\slim-records\emdb-slim-records\emdb-records-slim.ndjson...
----------------------------------------
STATISTICS (orjson)
----------------------------------------
Total records in EMDB file:   51645
Total records written:        51645
  - Found existing scores:    18132
  - Extrapolated scores:      33513
  - Skipped (no ID):          0
----------------------------------------
SUCCESS: Input count matches output count.


### Merge all FAIR score into one file

In [21]:
fair_scores_path = r"D:\pipeline-data\fair_scores\fair_scores.ndjson"

In [22]:
combine_ndjson_files([doi_fair_scores_path, emdb_fair_scores_path], fair_scores_path)

Lines processed: 49,000,000
Finished! Total entries saved: 49,061,167


## Topics

### Enhance fair scores from OpenAlex with subfiled, field, and domain

In [20]:
input_topics = r"D:\pipeline-data\topics\topics_oa\topics_only_oa.ndjson"
mapping_file = r"D:\pipeline-data\external\openalex-topics\openalex_topic_mapping_table.csv"
output_topics = r"D:\pipeline-data\topics\topics_oa\topics_oa.ndjson"

In [10]:
enhance_topics(input_topics, mapping_file, output_topics)

--- Starting Line-by-Line Enhancement ---
Loading mapping CSV...
Mapping loaded. 4,516 topics indexed.
Sample Key: 'T10001'
Processing lines...
Lines processed: 15,300,000 | Matches: 15,300,000

--- Done! ---
Total Lines: 15,324,819
Total Matches: 15,324,819
Saved to: D:\pipeline-data\topics\topics_enhanced.ndjson


### Standardize and group our topics assignment from our custom model

In [7]:
input_ndjson_path = r"D:\pipeline-data\topics\topics_custom_model\topics_files"
output_path = r"D:\pipeline-data\topics\topics_custom_model\topics_custom_model.ndjson"

In [8]:
restructure_topics_ndjson(input_ndjson_path, output_path)

Processing: 772 out of 772 files...
Processing complete
Total files processed: 772
Total lines in output: 49,061,167


## Dataset report

### Dataset metadata

#### Create metadata njson files so easier to load in table

In [3]:
slim_folder = r"D:\pipeline-data\records\slim-records"
dst_folder = r"D:\pipeline-data\dataset_index\metadata-records"

In [4]:
batch_process_metadata_from_slim(slim_folder, dst_folder)

Processing 772 files using 32 cores...
Input: D:\pipeline-data\records\slim-records
Output: D:\pipeline-data\dataset_index\metadata-records
[772/772] files completed
Done. files=772 kept=49,061,167 bad=0 time=999.3s rate≈49,097/rec-per-sec


{'files_seen': 772,
 'records_read': 49061167,
 'records_kept': 49061167,
 'records_bad_json': 0,
 'output_dir': 'D:\\pipeline-data\\dataset_index\\metadata-records',
 'elapsed_sec': 999.26,
 'rate_rec_per_sec': 49097}

#### Load metadata in duckdb table

In [4]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"
metadata_folder = r"D:\pipeline-data\dataset_index\metadata-records"

In [14]:
create_metadata_table(dataset_reports_db, metadata_folder)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Metadata table created in 'D:\pipeline-data\dataset_index\dataset_reports.duckdb'. Total rows: 49061167

Sample rows


ModuleNotFoundError: No module named 'matplotlib'

In [19]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM metadata LIMIT 3").df())
con.close()

,dataset_id,pub_ts,pubyear,creators,title,source
0,10.4225/15/515b76f4dbf24,2011-01-01,2011,"[{""name"":""WESTWOOD, KAREN JILLIAN"",""name_type""...",Primary Production in the Sub-Antarctic and Po...,datacite
1,10.4225/15/515b7978e65a0,2010-01-01,2010,"[{""name"":""WESTWOOD, KAREN JILLIAN"",""name_type""...","Primary productivity, pulse amplitude modulate...",datacite
2,10.1594/pangaea.808335,2012-01-01,2012,"[{""name"":""Glas, Martin S""},{""name"":""Langer, Ge...",(Figure 4) pH and Ca**2+ dynamics of an adult ...,datacite


### Citations, Mentions, FAIR scores, and Topics

In [6]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"
citations_file =  r"D:\pipeline-data\citations\citations.ndjson"
mentions_file =  r"D:\pipeline-data\mentions\mentions.ndjson"
fair_scores_file =  r"D:\pipeline-data\fair_scores\fair_scores.ndjson"

#### Load citations to duckdb

In [7]:
create_citations_table(dataset_reports_db, citations_file)

Loading Citations from: D:\pipeline-data\citations\citations.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Citations table created. Rows: 7,669,263

Preview
        dataset_id     cit_ts  citation_year  citation_weight  \
0  10.5517/cct09bf 2010-05-25           2010             1.11   
1  10.5517/ccst4mb 2010-05-25           2010             1.11   
2  10.5517/ccv7428 2010-05-25           2010             1.11   
3  10.5517/ccv7439 2010-05-25           2010             1.11   
4  10.5517/ccv744b 2010-05-26           2010             1.11   

               source  
0  ["datacite","mdc"]  
1  ["datacite","mdc"]  
2  ["datacite","mdc"]  
3  ["datacite","mdc"]  
4             ["mdc"]  


#### Load mentions to duckdb

In [32]:
create_mentions_table(dataset_reports_db, mentions_file)

Loading Mentions from: D:\pipeline-data\mentions\mentions.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Mentions table created. Rows: 4,156,510
        dataset_id     men_ts  mention_year  mention_weight   source
0  10.5517/cct09bf 2010-05-26          2010            1.11  ["mdc"]
1  10.5517/ccst4mb 2010-05-26          2010            1.11  ["mdc"]
2  10.5517/ccv7428 2010-05-26          2010            1.11  ["mdc"]
3  10.5517/ccv7439 2010-05-26          2010            1.11  ["mdc"]
4  10.5517/ccv744b 2010-05-26          2010            1.11  ["mdc"]


#### Load FAIR scores to duckdb

In [23]:
create_fair_scores_table(dataset_reports_db, fair_scores_file)

Loading FAIR Scores from: D:\pipeline-data\fair_scores\fair_scores.ndjson


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FAIR Scores table created. Rows: 49,061,167

Preview
        dataset_id  score
0  10.5284/1000389  30.77
1  10.5284/1000140  30.77
2  10.5284/1000146  50.00
3  10.5284/1000144  30.77
4  10.5284/1000181  30.77


#### Load topics to duckdb

##### Load OpenAlex topics table

In [ ]:
oa_file = r"D:\pipeline-data\topics\topics_oa\topics_oa.ndjson"
con = duckdb.connect(dataset_reports_db)
con.execute(f"""
        CREATE OR REPLACE TABLE topics_oa AS 
        SELECT * FROM read_json_auto('{oa_file}', ignore_errors=true);
    """)
display(con.execute("SELECT * FROM topics_oa LIMIT 3").df())
con.close()

##### Load custom model topics table

In [16]:
custom_file = r"D:\pipeline-data\topics\topics_custom_model\topics_custom_model.ndjson"
con = duckdb.connect(dataset_reports_db)
con.execute(f"""
        CREATE OR REPLACE TABLE topics_custom_model AS 
        SELECT * FROM read_json_auto('{custom_file}', ignore_errors=true);
    """)
display(con.execute("SELECT * FROM topics_custom_model LIMIT 3").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,10.5284/1000389,T13714,Medieval Architecture and Archaeology,0.5112,custom_model,1204,Archeology,12,Arts and Humanities,2,Social Sciences,None,None,None
1,10.5284/1000140,T10087,Archaeology and ancient environmental studies,0.5078,custom_model,1911,Paleontology,19,Earth and Planetary Sciences,3,Physical Sciences,None,None,None
2,10.5284/1000146,T10889,Soil erosion and sediment transport,0.3552,custom_model,1111,Soil Science,11,Agricultural and Biological Sciences,1,Life Sciences,None,None,None


In [21]:
con = duckdb.connect(dataset_reports_db)
query = """
SELECT 
    count(*) FILTER (WHERE topic_id IS NULL) AS missing_count,
    count(*) AS total_rows
FROM topics_custom_model;
"""
print("--- Missing Values Count ---")
print(con.execute(query).df())

# 2. Previewing the rows
preview_query = """
SELECT * FROM topics_custom_model
WHERE topic_id IS NULL
LIMIT 5;
"""
display(con.execute(preview_query).df())

con.close()

--- Missing Values Count ---
   missing_count  total_rows
0          25561    49061167


,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,10.5063/aa/pstango.3.1,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
1,10.5063/aa/wliao.103.1,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
2,10.5063/aa/wliao.103.2,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
3,10.14457/psu.res.2009.5,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None
4,10.14457/kku.res.2011.3,None,Unclassified,0.0,custom_model,None,Unclassified,None,Unclassified,None,Unclassified,None,None,None


##### Create final topics table (OpenAlex if exist and score >0.5 else custom if score> than OA or OA not exist)

In [17]:
create_topics_table(dataset_reports_db)

Creating final 'topics' table with score comparison logic


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Table created in 145.33 seconds.
Final 'topics' table contains 49,061,167 rows.

Sample of rows where Custom Model won:
               dataset_id        source   score
0  10.5287/bodleian6zr1.2  custom_model  0.2431
1  10.5287/bodleian8irf.2  custom_model  0.2526
2  10.5287/bodleian2pyz.2  custom_model  0.2824
3  10.5287/bodleian6ktj.2  custom_model  0.2702
4  10.5287/bodleiandmdw.2  custom_model  0.2264


In [20]:
# Export topics for our cloud database
con = duckdb.connect(dataset_reports_db)
output_file =  r"D:\pipeline-data\topics\topics.ndjson"
con.execute(f"""
    COPY topics 
    TO '{output_file}' 
    (FORMAT JSON, ARRAY FALSE);
""")
con.close()

with open(output_file, 'rb') as f:
    line_count = sum(1 for line in f)

print(f"Export complete: {output_file}")
print(f"Total lines in file: {line_count}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Export complete: D:\pipeline-data\topics\topics_custom_model\topics.ndjson
Total lines in file: 49061167


In [22]:
def split_ndjson(input_file, target_folder, lines_per_file=500000):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
    
    file_idx = 1
    line_count = 0
    out_file = None

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line_count % lines_per_file == 0:
                    if out_file:
                        out_file.close()
                    
                    chunk_path = os.path.join(target_folder, f'topics_part_{file_idx}.ndjson')
                    out_file = open(chunk_path, 'w', encoding='utf-8')
                    file_idx += 1
                
                out_file.write(line)
                line_count += 1
    finally:
        if out_file:
            out_file.close()

    print(f"Finished! Split {line_count} lines into {file_idx - 1} files.")

topics_file =  r"D:\pipeline-data\topics\topics.ndjson"
topics_split_folder =  r"D:\pipeline-data\topics\topics_split"
split_ndjson(topics_file, topics_split_folder)

Finished! Split 49061167 lines into 99 files.


In [18]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM topics WHERE source='openalex' LIMIT 3").df())
con.close()

,dataset_id,topic_id,topic_name,score,source,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name
0,10.5287/bodleiandi89.2,T11150,Endoplasmic Reticulum Stress and Disease,0.734048,openalex,1307,Cell Biology,13,"Biochemistry, Genetics and Molecular Biology",1,Life Sciences
1,10.5287/bodleiannfy.2,T10513,Natural Fiber Reinforced Composites,0.253378,openalex,2507,Polymers and Plastics,25,Materials Science,3,Physical Sciences
2,10.5287/bodleian8otp.2,T10521,RNA and protein synthesis mechanisms,0.236322,openalex,1312,Molecular Biology,13,"Biochemistry, Genetics and Molecular Biology",1,Life Sciences


## Create Master Dataset Metrics table (regroup everything)

In [3]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

In [4]:
create_dataset_metrics_table(dataset_reports_db)

Creating dataset_metrics table (topic, creators, FAIR score, 3-year metrics, etc. for each dataset)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! dataset_metrics table created. Total datasets: 49,061,167

Preview
           dataset_id                                           creators  \
0  10.15468/dl.wp7fxw  [{"name":"GBIF.Org User","name_type":"Organiza...   
1     10.15468/nya36c  [{"name":"2, Caitlin M. Carlson 2, Humberto E....   
2  10.15468/dl.0ysvg9  [{"name":"GBIF.Org User","name_type":"Organiza...   
3  10.15468/dl.fatfbv  [{"name":"GBIF.Org User","name_type":"Organiza...   
4  10.15468/dl.rbyoph  [{"name":"GBIF.Org User","name_type":"Organiza...   

                                          topic_name  cit_3yr  
0                        Data Analysis and Archiving        0  
1  N-Heterocyclic Carbenes in Organic and Inorgan...        0  
2                        Ga2O3 and related materials        0  
3                        Ga2O3 and related materials        0  
4              Information Retrieval and Data Mining        0  


In [4]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM dataset_metrics LIMIT 3").df())
con.close()

,dataset_id,pubyear,creators,dataset_source,topic_id,topic_name,topic_score,subfield_id,subfield_name,field_id,...,domain_name,fair_score,total_citations,total_cit_weight,cit_3yr,cit_weight_3yr,total_mentions,total_men_weight,men_3yr,men_weight_3yr
0,10.5281/zenodo.10347789,2019,"[{""name"":""WirtualneMuzeaMalopolski"",""name_type...",datacite,T14055,Consumer Packaging Perceptions and Trends,0.8692,1406,Marketing,14,...,Social Sciences,13.46,0,0.0,0,0.0,0,0.0,0,0.0
1,10.5281/zenodo.10347790,2019,"[{""name"":""WirtualneMuzeaMalopolski"",""name_type...",datacite,T14055,Consumer Packaging Perceptions and Trends,0.8692,1406,Marketing,14,...,Social Sciences,51.92,0,0.0,0,0.0,0,0.0,0,0.0
2,10.5281/zenodo.10347793,2022,"[{""name"":""hsjapan3dcg"",""name_type"":""Personal"",...",datacite,T12846,Ginkgo biloba and Cashew Applications,0.5917,2707,Complementary and alternative medicine,27,...,Health Sciences,51.92,0,0.0,0,0.0,0,0.0,0,0.0


## Normalization factors

In [ ]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

### Create normalization factors by topics table

In [26]:
calculate_normalization_factors_topics(dataset_reports_db)

Creating normalization_factors_topics table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Generating Global Benchmark...
   Generating benchmarks for 4516 topics...
Generating rolling medians for target years...
Saving 343292 benchmark rows...
normalization_factors_topics table created.

--- Sample view (Lifetime Topic Metrics) ---
  topic_id                               topic_name  median_cit_weight_3yr  \
0   T10001      Geological and Geochemical Analysis                    0.0   
1   T10002        Advanced Chemical Physics Studies                    0.0   
2   T10003      Innovation and Knowledge Management                    0.0   
3   T10004        Soil Carbon and Nitrogen Dynamics                    0.0   
4   T10005  Ecology and Vegetation Dynamics Studies                    0.0   

   n_cit  
0   9683  
1   2596  
2    861  
3   2770  
4   2781  


In [27]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_topics WHERE n_cit > 20 ORDER BY median_cit_weight_3yr DESC LIMIT 5").df())
con.close()

,topic_id,topic_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,T10043,Substance Abuse Treatment and Outcomes,2000,13.46,1.45,0.00,28,25,25
1,T11719,Data Quality and Management,2018,15.38,1.27,1.27,599,468,468
2,T14187,Varied Academic Research Topics,2018,15.38,1.27,1.27,35,33,33
3,T11719,Data Quality and Management,2019,15.38,1.27,1.27,668,458,458
4,T14187,Varied Academic Research Topics,2019,15.38,1.27,1.27,35,34,34


### Create normalization factors by subfield table

In [23]:
calculate_normalization_factors_subfields(dataset_reports_db)

Calculating Normalization Factors by Subfield (Guaranteed Benchmarks)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Generating Global Benchmark...
   Generating benchmarks for 252 subfields...
Generating rolling medians for target years...
Saving 19228 benchmark rows...
Normalization subfields table created.

--- Sample view (Lifetime Subfield Metrics) ---
  subfield_id                                 subfield_name  \
0        1100  General Agricultural and Biological Sciences   
1        1102                     Agronomy and Crop Science   
2        1103                    Animal Science and Zoology   
3        1104                               Aquatic Science   
4        1105  Ecology, Evolution, Behavior and Systematics   

   median_cit_weight_3yr   n_cit  
0                    0.0   24601  
1                    0.0   16848  
2                    0.0   11905  
3                    0.0   11362  
4                    0.0  126347  


In [25]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_subfields WHERE n_cit > 20 ORDER BY median_cit_weight_3yr DESC LIMIT 5").df())
con.close()

,subfield_id,subfield_name,pubyear,median_fair_score_3yr,median_cit_weight_3yr,median_men_weight_3yr,n_fair,n_cit,n_men
0,2505,Materials Chemistry,1998,13.46,1.06,1.06,4698,2446,2446
1,3106,Nuclear and High Energy Physics,2013,13.46,1.05,0.00,279,161,161
2,2505,Materials Chemistry,1999,13.46,1.04,1.04,7553,4343,4343
3,2505,Materials Chemistry,2017,13.46,1.03,1.04,133897,87675,87675
4,2505,Materials Chemistry,2018,13.46,1.01,1.00,142864,93009,93009


### Create normalization factors by subfield and raw Dataset Index

In [13]:
calculate_normalization_factors_subfields_rawDindex(dataset_reports_db)

Creating normalization_factors_subfields_rawDindex table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Generating All-time Benchmarks...
Generating yearly benchmarks for Target Years 1953 to 2026...
Saving 12279 benchmark rows...
normalization_factors_subfields_rawDindex table created.

-Preview (Top 5 Yearly Benchmarks) ---
                       subfield_name  target_year  median_raw_Dindex   n_count
0                                NaN         2026           0.044867  19866991
1  Statistical and Nonlinear Physics         2026           0.044867  10114122
2                                NaN         2024           0.051267   7537887
3                                NaN         2025           0.044867   3179198
4              Aerospace Engineering         2026           0.044867   2214913


In [14]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM normalization_factors_subfields_rawDindex WHERE pubyear=2020 LIMIT 10").df())
con.close()

,subfield_id,subfield_name,pubyear,median_raw_Dindex,n_count
0,NaN,NaN,2020,0.044867,970267
1,1100,General Agricultural and Biological Sciences,2020,0.044867,1456
2,1102,Agronomy and Crop Science,2020,0.044867,967
3,1103,Animal Science and Zoology,2020,0.044867,4010
4,1104,Aquatic Science,2020,0.102567,725
5,1105,"Ecology, Evolution, Behavior and Systematics",2020,0.044867,117859
6,1106,Food Science,2020,0.044867,3520
7,1107,Forestry,2020,0.044867,1982
8,1108,Horticulture,2020,0.044867,154
9,1109,Insect Science,2020,0.256400,1219


## Dataset Index

In [49]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

In [50]:
create_dataset_index_table(dataset_reports_db)

Initializing dataset_index creation


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Table 'dataset_index' created.
Total Rows: 49,061,167
Execution Time: 912.32 seconds


In [88]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT pubyear, topic_id, subfield_id, fair_score, t_norm_fair_final, total_cit_weight, t_norm_cit_final, total_men_weight, t_norm_men_final, dataset_index_topic, FROM dataset_index WHERE t_norm_cit_final > 1 ORDER BY total_cit_weight DESC LIMIT 5").df())
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pubyear,topic_id,subfield_id,fair_score,t_norm_fair_final,total_cit_weight,t_norm_cit_final,total_men_weight,t_norm_men_final,dataset_index_topic
0,1991,T12174,2711,69.23,20.00,2554.67,5.305,2554.67,1.00,1013.230136
1,1984,T10443,3320,13.46,20.00,821.70,17.790,821.70,17.79,31.016913
2,1979,T13122,1203,13.46,73.08,573.28,1.110,385.18,1.11,287.887220
3,2009,T14445,2205,13.46,20.00,564.32,11.890,518.94,11.93,30.544492
4,2002,T12795,1110,13.46,34.62,561.03,1.030,402.78,1.03,312.042219


## S-index

In [3]:
dataset_reports_db = r"D:\pipeline-data\dataset_index\dataset_reports.duckdb"

### Create a creators_table first exploding the dataset_index table on creators first

In [74]:
create_creators_table(dataset_reports_db)

FULL RUN: Creating 'creators_table' (Exploded with Context & Raw Metrics)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--------------------------------------------------
Success! 'creators_table' created.
Total Exploded Rows: 216,688,512
Execution Time: 13301.18 seconds
--------------------------------------------------

Preview of Identifier Normalization:
                 creator_name   primary_identifier
0    Tschonghongei, Nelson C.  0009-0007-9932-8135
1    Tschonghongei, Nelson C.  0009-0007-9932-8135
2  Ramos-Ordoñez, María Felix  0000-0002-9470-6375
3  Ramos-Ordoñez, María Felix  0000-0002-9470-6375
4  Ramos-Ordoñez, María Felix  0000-0002-9470-6375


In [75]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM creators_table ORDER BY total_cit_weight DESC LIMIT 5").df())
con.close()

,dataset_id,pubyear,topic_id,topic_name,subfield_id,subfield_name,creator_name,name_type,primary_identifier,affiliations,dataset_index_topic,dataset_index_subfield,total_cit_weight,total_men_weight,fair_score,total_citations,total_mentions
0,10.5519/0002965,2014,T11986,Scientific Computing and Data Management,1802,Information Systems and Management,Natural History Museum,Organizational,https://ror.org/039zvsn29,NaN,11885.491333,11885.491333,35541.60,111.22,73.08,21537,66
1,10.15468/ab3s5x,2025,T14423,Military Technology and Strategies,2202,Aerospace Engineering,iNaturalist contributors,NaN,NaN,"[""iNaturalist""]",10388.819000,10388.819000,30950.69,212.69,61.54,30948,210
2,10.15468/hnhrg3,2025,T10895,Species Distribution and Climate Change,2302,Ecological Modeling,Informatics and Data Science Center-Digital St...,NaN,NaN,"[""National Museum of Natural History, Smithson...",15709.769667,15709.769667,28524.52,18601.52,65.38,28524,18601
3,10.15468/hnhrg3,2025,T10895,Species Distribution and Climate Change,2302,Ecological Modeling,"Orrell, Thomas",Personal,0000-0003-1038-3028,"[""National Museum of Natural History, Smithson...",15709.769667,15709.769667,28524.52,18601.52,65.38,28524,18601
4,10.15468/ib5ypt,2025,T12568,Plant Taxonomy and Phylogenetics,1105,"Ecology, Evolution, Behavior and Systematics","Creuwels, Jeroen",Personal,0000-0001-6131-7026,"[""Naturalis Biodiversity Center""]",13300.269667,13300.269667,28171.27,11726.27,65.38,28171,11726


### Create S-index table by matching identifiers only

In [79]:
create_s_index_identifier_table(dataset_reports_db)

Initializing S_index_identifier (Master Researcher Profile)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--------------------------------------------------
Success! S_index_identifier created.
Total Unique Researchers based on identifier: 441,944
Execution Time: 358.95 seconds
--------------------------------------------------

Top 5 Researchers (with Topic/Subfield IDs):
                           primary_identifier  \
0                         0000-0001-5473-2109   
1  https://nrid.nii.ac.jp/nrid/1000050260047/   
2                   https://ror.org/0566bfb96   
3                         0000-0002-9160-682x   
4  https://nrid.nii.ac.jp/nrid/1000040300727/   

                     primary_topic_name            primary_subfield_name  \
0     Geochemistry and Geologic Mapping          Artificial Intelligence   
1  Magnetic confinement fusion research  Nuclear and High Energy Physics   
2     Geochemistry and Geologic Mapping          Artificial Intelligence   
3    Prenatal Screening and Diagnostics                Molecular Biology   
4  Magnetic confinement fusion research  Nuclear and Hi

In [113]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_identifier LIMIT 5").df())
con.close()

,primary_identifier,creator_names,name_type,all_affiliations,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,0000-0001-5473-2109,"[TOKUZAWA, Tokihiko]",Personal,"[National Institute for Fusion Science, Nation...",T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,4499,252,...,5727142,1.393599e+06,1.400136e+06,0.243332,0.244474,0.00,0.00,0.0,0.0,14.673781
1,https://nrid.nii.ac.jp/nrid/1000050260047/,"[TANAKA, Kenji]",Personal,"[National Institute for Fusion Science, Nation...",T10346,Magnetic confinement fusion research,3106,Nuclear and High Energy Physics,4491,252,...,3208893,7.884570e+05,7.903823e+05,0.245710,0.246310,0.00,0.00,0.0,0.0,14.785334
2,https://ror.org/0566bfb96,"[Naturalis Biodiversity Center, Distributed Sy...",Organizational,[],T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,4255,251,...,2388894,6.805641e+05,6.805869e+05,0.284887,0.284896,1.13,1.13,1.0,1.0,17.094411
3,0000-0002-9160-682x,"[GOTO, Motoshi]",Personal,"[National Institute for Fusion Science (NIFS),...",T10978,Prenatal Screening and Diagnostics,1312,Molecular Biology,4504,252,...,2608151,6.344953e+05,6.364951e+05,0.243274,0.244041,0.00,0.00,0.0,0.0,14.659584
4,https://nrid.nii.ac.jp/nrid/1000040300727/,"[FUNABA, Hisamichi]",Personal,"[National Institute for Fusion Science (NIFS),...",T10346,Magnetic confinement fusion research,3106,Nuclear and High Energy Physics,4490,252,...,2104401,6.296659e+05,6.357325e+05,0.299214,0.302097,276241.20,0.00,197006.0,0.0,15.511266


### Create S-index table by matching name and affiliations set

In [5]:
create_s_index_name_affiliation_table(dataset_reports_db)

Creating s_index_name_affiliation table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! 's_index_name_affiliation' created.
Total Unique Name-Affiliation Sets: 3,962,943
Execution Time: 289.25 seconds

Preview Top 5 Rows:
          grouping_name affiliation_set_signature  n_datasets  S_index_topics
0    nilsson, r. henrik                      None     3201095    1.619952e+06
1      abarenkov, kessy                      None     3201073    1.619924e+06
2        kõljalg, urmas                      None     3201065    1.619920e+06
3  larsson, karl-henrik                      None     3200948    1.619337e+06
4        tedersoo, leho                      None     2383786    1.224757e+06


In [6]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT * FROM S_index_name_affiliation LIMIT 5").df())
con.close()

,grouping_name,affiliation_set_signature,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,n_unique_topics,n_unique_subfields,first_pub_year,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,"nilsson, r. henrik",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2012,...,3201095,1.619952e+06,1.627984e+06,0.506062,0.508571,864.65,814.86,625.0,590.0,31.049449
1,"abarenkov, kessy",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2012,...,3201073,1.619924e+06,1.627956e+06,0.506056,0.508566,862.65,812.86,623.0,588.0,31.049163
2,"kõljalg, urmas",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2015,...,3201065,1.619920e+06,1.627953e+06,0.506057,0.508566,864.87,815.08,625.0,590.0,31.049142
3,"larsson, karl-henrik",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4477,252,2015,...,3200948,1.619337e+06,1.627369e+06,0.505893,0.508402,57.90,57.90,43.0,43.0,31.049087
4,"tedersoo, leho",None,Personal,T10451,Mycorrhizal Fungi and Plant Interactions,1110,Plant Science,4471,252,2015,...,2383786,1.224757e+06,1.228910e+06,0.513787,0.515529,2009.15,69.15,1992.0,53.0,30.919935


### Create S-index table by matching identifier first then name/affiliation

In [14]:
create_s_index_identifier_name_affiliation_table(dataset_reports_db)

Creating s_index_identifier_name_affiliation table


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Completed in 1718.90 seconds.
Total authors: 3,913,588
  > By Identifier: 441,944
  > By name & affiliation: 3,471,644


In [7]:
con = duckdb.connect(dataset_reports_db)
display(con.execute("SELECT* from s_index_identifier_name_affiliation LIMIT 5").df())
con.close()

,distinct_group_id,grouping_method,display_name,primary_identifier,all_affiliations,name_type,primary_topic_id,primary_topic_name,primary_subfield_id,primary_subfield_name,...,n_datasets,S_index_topics,S_index_subfield,avg_dataset_index_topics,avg_dataset_index_subfield,total_cit_weight,total_men_weight,sum_total_citations,sum_total_mentions,avg_fair_score
0,gbif.org user_[],name_affiliation,GBIF.org User,NaN,[],Organizational,T10895,Species Distribution and Climate Change,1312,Molecular Biology,...,891,205.109293,205.266333,0.230201,0.230377,0.0,0.0,0.0,0.0,13.822649
1,andr\xc3\xa9s chirinos_[],name_affiliation,Andr\xC3\xA9s Chirinos,NaN,[],NaN,T11594,Tree-ring climate responses,1902,Atmospheric Science,...,2,2.820667,2.820667,1.410333,1.410333,0.0,0.0,0.0,0.0,84.620000
2,"p\xc3\xa9rez expolio, jes\xc3\xbas_[]",name_affiliation,"P\xC3\xA9rez Expolio, Jes\xC3\xBAs",NaN,[],Personal,T10978,Prenatal Screening and Diagnostics,2735,"Pediatrics, Perinatology and Child Health",...,2,2.564000,2.564000,1.282000,1.282000,0.0,0.0,0.0,0.0,76.920000
3,0009-0001-6783-2245,identifier,Seyed Arsham Asgari,0009-0001-6783-2245,[\x22Independent Researcher\x22],Personal,T11135,Virology and Viral Diseases,2713,Epidemiology,...,2,2.179500,2.179500,1.089750,1.089750,0.0,0.0,0.0,0.0,65.385000
4,0000-0002-9470-6375,identifier,"Ramos-Ordo\xC3\xB1ez, Mar\xC3\xADa Felix",0000-0002-9470-6375,[\x22Universidad Nacional Aut\xC3\xB3noma de M...,Personal,T12157,Geochemistry and Geologic Mapping,1702,Artificial Intelligence,...,3,1.538500,1.538500,0.512833,0.512833,0.0,0.0,0.0,0.0,30.770000
